<a href="https://colab.research.google.com/github/ssprajapati2021/Hybrid-RAG-Fine-Tuning/blob/main/notebook/Data_Understanding_and_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 1: Data Understanding & Exploratory Data Analysis**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED for this notebook:**
- [ ] `corporate_policies/` folder with `.md` SOP files uploaded to Colab:
  - `refund_policy.md`, `shipping_delays.md`, `password_reset.md`, `technical_troubleshooting.md`, `escalation_matrix.md`
- [ ] Internet access (to download Bitext dataset from HuggingFace)

**Files this notebook will CREATE:**
- [ ] `sampled_data.csv` — Cleaned, deduplicated, 4000-row sample of the Bitext dataset
  _(Required by: Notebook 2 — Data Preparation)_

> **Note:** Tasks 1.1 (Define Project Scope) and 1.4 (Design Methodology) are documented in your **Project Proposal PDF**, not this notebook. This notebook covers Tasks **1.2** and **1.3**.

---

## **Stage 1: Data Understanding & EDA**
### **Task 1.2: Understand and Assess Data**

#### **1.2.1 Inspect Dataset Structure [2 marks]**
**The Task:** Identify the presence of the corporate policy files within the local environment and successfully download the `bitext` customer support dataset from Hugging Face. Print the dataset information.

**Hints & Tips:**
* Use `glob` to find `.md` files in your workspace, and `load_dataset` for the bitext data.
* The dataset ID is `bitext/Bitext-customer-support-llm-chatbot-training-dataset` — it contains ~26,872 rows.
* Key columns: `instruction` (user query), `intent` (target class), `category` (broader group).

**Why we are doing it:** To ensure the local environment is properly configured with our knowledge base before reading data.
**How we are doing it:** Check the `corporate_policies` directory, load the dataset, and use `.info()`.

**Learner Inference:** If the files load correctly, your environment is stable. Seeing the 'intent' column confirms the target variable you will train your model to extract in Stage 4.

In [7]:
# Find .md files
import glob
import pandas as pd
md_files=glob.glob('*.md')
#print(md_files)
df = pd.DataFrame({"Markdown Files": md_files})
display(df)

,Markdown Files
0,data_privacy.md
1,password_reset.md
2,payment_methods.md
3,working_hours.md
4,account_recovery.md
5,shipping_delays.md
6,product_return.md
7,technical_troubleshooting.md
8,billing_disputes.md
9,refund_policy.md


In [9]:
#Load the dataset from Hugging Face
from datasets import load_dataset

ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

print(ds)

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})


#### Inspecting the Training Dataset

**Why are we doing this?**

The Hugging Face `load_dataset()` function returns a `DatasetDict`, which is optimized for machine learning workflows but does not provide Pandas' rich exploratory data analysis (EDA) capabilities. To perform detailed data inspection, we convert the training split into a Pandas DataFrame.

**How are we doing it?**

1. Extract the **train** split from the `DatasetDict`.
2. Convert the Hugging Face `Dataset` into a Pandas DataFrame.
3. Use the DataFrame's `.info()` method to examine the dataset structure, including:
   - Total number of records
   - Column names
   - Data types
   - Non-null value counts
   - Memory usage

This initial inspection helps verify that the dataset has been loaded correctly and is ready for further preprocessing, visualization, and model fine-tuning.

In [11]:
#Load Train DataSet as load_dataset providing Dataset Dictionary with train key
train_ds = ds["train"]

#Convert the Hugging Face Dataset to a Pandas DataFrame for EDA
df = train_ds.to_pandas()

#Display dataset structure and summary statistics
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26872 entries, 0 to 26871
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   flags        26872 non-null  object
 1   instruction  26872 non-null  object
 2   category     26872 non-null  object
 3   intent       26872 non-null  object
 4   response     26872 non-null  object
dtypes: object(5)
memory usage: 1.0+ MB


#### **1.2.2 Quantify Dataset Characteristics [3 marks]**
**The Task:** Calculate the original size of the dataset, identify the number of unique intents, and randomly sample the dataset down to 4,000 rows.

**Hints & Tips:**
* Use `len()`, `.nunique()`, and `.sample(4000, random_state=42)`.
* `random_state=42` ensures reproducibility — everyone gets the same sample.
* With 4000 rows and 27 intents, you'll get ~148 examples per intent on average.

**Parameter Tuning:**
* `sample(n)` — try 1000–5000 per the workflow spec:
  - `1000`: Fast (~1 min), lower accuracy
  - `4000`: Moderate (~3 min), better accuracy (recommended)
  - `5000`: Slower, best accuracy, may hit T4 memory limits

**Learner Inference:** Capping at 4000 rows keeps your Stage 4 training loop to minutes while providing enough examples per intent.

In [17]:
# Original DataSet size
original_size=len(df)

# Number of unique intents
unique_intents=df['intent'].nunique()

# Randomly sample 4,000 records
sampled_df = df.sample(n=4000, random_state=42)

print(f"Original Dataset Size: {original_size}")
print(f"\nUnique Intents: {unique_intents}")
print(f"\nSampled Dataset Size: \n{sampled_df}")

Original Dataset Size: 26872

Unique Intents: 27

Sampled Dataset Size: 
       flags                                        instruction category  \
9329     BLZ                   I can't talk with  a human agent  CONTACT   
4160    BLMZ  I have got to locate hte bills from {{Person N...  INVOICE   
18500  BCELM  I cannot pay, help me to inform of a problem w...  PAYMENT   
8840      BL           I want help speaking to customer service  CONTACT   
5098     BLZ           I try to see th accepted payment options  PAYMENT   
...      ...                                                ...      ...   
5955      BL  I want to check in which situations can I ask ...   REFUND   
20554  BILQZ   how cab i find information about my key recovery  ACCOUNT   
19010    BLM            help me to acquire some of your article    ORDER   
4520    BLMZ                   see my blls from {{Person Name}}  INVOICE   
18057  BILMQ       where can i notify of troubles with payments  PAYMENT   

              

#### **1.2.3 Validate Sample Quality [2 marks]**
**The Task:** Identify and remove records containing null values or duplicated customer instructions using normalised deduplication.

**Hints & Tips:**
* Use `.isnull().sum()` to check nulls.
* Simple `drop_duplicates` misses near-dupes like `"Where is my order?"` vs `"where is my order?"`.
* Normalise to lowercase + strip whitespace BEFORE deduplicating to catch these.
* The Bitext dataset typically has ~15 near-duplicates in a 4000-row sample.

**Learner Inference:** Cleaning now prevents "garbage-in, garbage-out". Duplicate prompts can cause leakage between train and test sets later.

In [22]:
# Check for missing values
null_counts = sampled_df.isnull().sum()

print("Missing values by column:")
print(null_counts)

print(f"\nTotal missing values: {null_counts.sum()}")

# Normalize customer instructions
sampled_df["instruction_normalized"] = (
    sampled_df["instruction"]
      .str.lower()
      .str.strip()
)

# Count duplicate instructions
count_duplicate = sampled_df.duplicated(subset="instruction_normalized").sum()

print(f"\nDuplicate instructions found: {count_duplicate}")

# Remove duplicates based on normalized instructions
sampled_df = sampled_df.drop_duplicates(subset="instruction_normalized")

print(f"Dataset size after removing duplicates: {len(sampled_df)}")

# Remove the temporary column
sampled_df = sampled_df.drop(columns="instruction_normalized")

Missing values by column:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Total missing values: 0

Duplicate instructions found: 98
Dataset size after removing duplicates: 3902


### **Task 1.3: Perform Exploratory Data Analysis**

#### **1.3.1 Perform Univariate Analysis [2 marks]**
**The Task:** Plot the top 10 most frequent customer intents using a horizontal bar chart.

**Hints & Tips:**
* Use a Seaborn `countplot` with `order=df['intent'].value_counts().index[:10]`.
* Check if any single intent dominates >80% of data — signals severe class imbalance.
* `palette="viridis"` works well for accessibility.

**Learner Inference:** This visual reveals class imbalance. If one intent dominates, your fine-tuned router becomes biased toward guessing it.

In [ ]:
# YOUR CODE HERE


#### **1.3.2 Perform Bivariate Analysis [2 marks]**
**The Task:** Read the text from the uploaded Markdown SOP files and fit a `TfidfVectorizer` (with `stop_words='english'`) to transform them into vectors.

**Hints & Tips:**
* Read files into a list of strings first using `open(f, 'r').read()`.
* `TfidfVectorizer(stop_words="english")` removes common words, keeping meaningful terms.
* The resulting matrix shape should be `(num_docs, num_unique_terms)`.

**Learner Inference:** This transforms raw text into math, letting you prove whether your policies are distinct enough for an AI to tell apart.

In [ ]:
# YOUR CODE HERE


#### **1.3.3 Generate Visualisations [2 marks]**
**The Task:** Calculate cosine similarity between the TF-IDF vectors of the SOP documents and plot an annotated heatmap.

**Hints & Tips:**
* Use `cosine_similarity` from sklearn and `sns.heatmap` with `annot=True`.
* Diagonal should always be 1.0 (document vs itself).
* Off-diagonal >0.5 = high overlap (RAG may struggle); <0.3 = low overlap (good retrieval).

**Learner Inference:** High overlap (dark off-diagonal squares) means the RAG system struggles to fetch the right doc. Low overlap proves retrieval will be accurate.

In [ ]:
# YOUR CODE HERE


---
## Save Artifacts for Notebook 2

**IMPORTANT:** This cell saves the cleaned DataFrame. Notebook 2 depends on this file.

In [ ]:
# YOUR CODE HERE


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify all items before proceeding to Notebook 2.**

- [ ] Bitext dataset loaded and `.info()` printed
- [ ] Dataset sampled to 4000 rows (or chosen size 1000–5000)
- [ ] Nulls inspected, near-duplicates removed via normalised dedup
- [ ] Top 10 intent bar chart rendered
- [ ] TF-IDF vectorisation completed on SOP documents
- [ ] Cosine similarity heatmap rendered
- [ ] **`sampled_data.csv` saved to disk** ← _CRITICAL for Notebook 2_
- [ ] `corporate_policies/` folder still accessible ← _Needed in Notebooks 2 and 4_

**If any item is unchecked, fix it before moving on.**